In [101]:
# Loading the dataset in - since it is so large, it must be done using batch processing

import pandas as pd
import numpy as np
import kagglehub


csv_path = kagglehub.dataset_download(
    "sobhanmoosavi/us-accidents",
    path="US_Accidents_March23.csv",
)

batch_size = 10000
batches = []
batch_num = 0

for batch in pd.read_csv(csv_path, chunksize=batch_size, low_memory=False):
    batches.append(batch)
    batch_num += 1
    if batch_num % 20 == 0:
        print(f"Processed {batch_num * batch_size} rows")

df = pd.concat(batches, ignore_index=True)
print(df.shape)

Processed 200000 rows
Processed 400000 rows
Processed 600000 rows
Processed 800000 rows
Processed 1000000 rows
Processed 1200000 rows
Processed 1400000 rows
Processed 1600000 rows
Processed 1800000 rows
Processed 2000000 rows
Processed 2200000 rows
Processed 2400000 rows
Processed 2600000 rows
Processed 2800000 rows
Processed 3000000 rows
Processed 3200000 rows
Processed 3400000 rows
Processed 3600000 rows
Processed 3800000 rows
Processed 4000000 rows
Processed 4200000 rows
Processed 4400000 rows
Processed 4600000 rows
Processed 4800000 rows
Processed 5000000 rows
Processed 5200000 rows
Processed 5400000 rows
Processed 5600000 rows
Processed 5800000 rows
Processed 6000000 rows
Processed 6200000 rows
Processed 6400000 rows
Processed 6600000 rows
Processed 6800000 rows
Processed 7000000 rows
Processed 7200000 rows
Processed 7400000 rows
Processed 7600000 rows
(7728394, 46)


In [102]:
# Finding the top 10 cities with the most accidents
df["City"].value_counts().head(10)

City
Miami          186917
Houston        169609
Los Angeles    156491
Charlotte      138652
Dallas         130939
Orlando        109733
Austin          97359
Raleigh         86079
Nashville       72930
Baton Rouge     71588
Name: count, dtype: int64

In [103]:
# To trim the dataset, we choose the top 3 cities with the most accidents: Miami, Houston, and Los Angeles
df_trimmed = df[df["City"].isin(["Miami", "Houston", "Los Angeles"])]
df_trimmed.head()

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
42866,A-42867,Source2,2,2016-06-21 10:46:30,2016-06-21 11:27:00,34.078926,-118.289040,NaN,NaN,0.0,Right hand shoulder blocked due to accident on...,US-101 N,Los Angeles,Los Angeles,CA,90004,US,US/Pacific,KCQT,2016-06-21 10:47:00,82.9,NaN,47.0,29.95,10.0,Variable,4.6,NaN,Clear,False,False,False,False,True,False,False,False,False,False,False,False,False,Day,Day,Day,Day
42867,A-42868,Source2,3,2016-06-21 10:49:21,2016-06-21 11:34:21,34.091179,-118.239471,NaN,NaN,0.0,Accident on I-5 Northbound at Exit 138 Stadium...,Golden State Fwy S,Los Angeles,Los Angeles,CA,90031,US,US/Pacific,KCQT,2016-06-21 10:47:00,82.9,NaN,47.0,29.95,10.0,Variable,4.6,NaN,Clear,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day
42881,A-42882,Source2,3,2016-06-21 10:51:45,2016-06-21 11:36:45,34.037239,-118.309074,NaN,NaN,0.0,Accident on I-10 Eastbound at Exit 10 Western ...,I-10 W,Los Angeles,Los Angeles,CA,90018,US,US/Pacific,KCQT,2016-06-21 10:47:00,82.9,NaN,47.0,29.95,10.0,Variable,4.6,NaN,Clear,False,False,False,False,False,False,False,False,True,False,False,False,False,Day,Day,Day,Day
42883,A-42884,Source2,3,2016-06-21 10:56:24,2016-06-21 11:34:00,34.027458,-118.274490,NaN,NaN,0.0,Accident on I-110 Northbound at Exit 20C Adams...,Harbor Fwy N,Los Angeles,Los Angeles,CA,90007,US,US/Pacific,KCQT,2016-06-21 10:47:00,82.9,NaN,47.0,29.95,10.0,Variable,4.6,NaN,Clear,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day
42898,A-42899,Source2,3,2016-06-21 11:30:46,2016-06-21 12:00:46,33.947544,-118.279434,NaN,NaN,0.0,Right hand shoulder blocked due to accident on...,Harbor Fwy N,Los Angeles,Los Angeles,CA,90003,US,US/Pacific,KHHR,2016-06-21 11:53:00,80.1,NaN,52.0,29.96,10.0,ESE,9.2,NaN,Clear,False,False,False,False,False,False,False,False,False,False,False,False,False,Day,Day,Day,Day


In [104]:
# select columns to be dropped
columns_dropped = [
    # zero/near-zero variance
    'Turning_Loop',
    'Roundabout',
    'No_Exit',
    'Traffic_Calming',
    'Bump',

    # redundant or irrelevant
    'Country',
    'State',
    'End_Lat',
    'End_Lng',
    'Airport_Code',
    'Weather_Timestamp',
    'County',
    'Timezone',
    'Civil_Twilight',
    'Nautical_Twilight',
    'Astronomical_Twilight',
    'Wind_Chill(F)',
    'ID',
    'Description',
    'Street',
]

# # create a copy to keep a primary df and apply transformation
d2 = df_trimmed.copy().drop(columns_dropped, axis=1)

print(f"Columns before: {df_trimmed.shape[1]}")
print(f"Columns after: {d2.shape[1]}")
print(f"Remaining columns: {list(d2.columns)}")

Columns before: 46
Columns after: 26
Remaining columns: ['Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'Distance(mi)', 'City', 'Zipcode', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Crossing', 'Give_Way', 'Junction', 'Railway', 'Station', 'Stop', 'Traffic_Signal', 'Sunrise_Sunset']


In [105]:
# checking for null values
d2.info()

<class 'pandas.core.frame.DataFrame'>
Index: 513017 entries, 42866 to 7728376
Data columns (total 26 columns):
 #   Column             Non-Null Count   Dtype  
---  ------             --------------   -----  
 0   Source             513017 non-null  object 
 1   Severity           513017 non-null  int64  
 2   Start_Time         513017 non-null  object 
 3   End_Time           513017 non-null  object 
 4   Start_Lat          513017 non-null  float64
 5   Start_Lng          513017 non-null  float64
 6   Distance(mi)       513017 non-null  float64
 7   City               513017 non-null  object 
 8   Zipcode            513017 non-null  object 
 9   Temperature(F)     507387 non-null  float64
 10  Humidity(%)        507117 non-null  float64
 11  Pressure(in)       509236 non-null  float64
 12  Visibility(mi)     508298 non-null  float64
 13  Wind_Direction     506361 non-null  object 
 14  Wind_Speed(mph)    471323 non-null  float64
 15  Precipitation(in)  371056 non-null  float64
 16  We

## Null Value Handling

In [106]:
# populating null values instead of completely dropping using forward fill (time-series) with a 3 hour gap for similar results (can't use KNN because data too large)

import numpy as np

# sort first
d2 = d2.sort_values(['City', 'Start_Time'])

# change to datetime (valuerror showed to add in format='ISO8601' as a parameter)
d2['Start_Time'] = pd.to_datetime(d2['Start_Time'], format='ISO8601')
d2['End_Time'] = pd.to_datetime(d2['End_Time'], format='ISO8601')

# select weather columns
weather_cols = ['Temperature(F)', 'Humidity(%)', 'Pressure(in)', 
                'Visibility(mi)', 'Wind_Direction', 
                'Wind_Speed(mph)', 'Weather_Condition']

# tweak this threshold as needed, currently 3 hours
max_gap = pd.Timedelta(hours=3)

# forward fill weather values, but only if the time gap between the current row and the last known value is within the max_gap threshold
def ffill_with_time_limit(group, cols, max_gap):
    for col in cols:
        filled = group[col].ffill()

        # calculate time diff between each row and the last non-null row
        last_valid_time = group['Start_Time'].where(group[col].notna()).ffill()
        time_gap = group['Start_Time'] - last_valid_time

        # only accept the fill if the gap is within threshold
        group[col] = filled.where(time_gap <= max_gap, other=np.nan)
    return group

# apply the function to each city group (done by city to avoid filling across different cities which may have different weather conditions)
city_groups = []
for city, group in d2.groupby('City'):
    filled_group = ffill_with_time_limit(group, weather_cols, max_gap)
    city_groups.append(filled_group)

# concatenate the groups back together
d2 = pd.concat(city_groups).sort_values(['City', 'Start_Time'])

# precipitation still just fills with 0
d2['Precipitation(in)'] = d2['Precipitation(in)'].fillna(0)

# check remaining nulls
print(d2.isnull().sum())

Source                  0
Severity                0
Start_Time              0
End_Time                0
Start_Lat               0
Start_Lng               0
Distance(mi)            0
City                    0
Zipcode                 0
Temperature(F)        625
Humidity(%)           634
Pressure(in)          572
Visibility(mi)        595
Wind_Direction        611
Wind_Speed(mph)      9566
Precipitation(in)       0
Weather_Condition     586
Amenity                 0
Crossing                0
Give_Way                0
Junction                0
Railway                 0
Station                 0
Stop                    0
Traffic_Signal          0
Sunrise_Sunset          6
dtype: int64


In [107]:
# drop the remaining null columns
d2 = d2.dropna()
# reset the index for simplicity
d2.reset_index(drop=True)

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,Zipcode,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset
0,Source2,2,2016-06-14 20:21:49,2016-06-14 21:07:49,29.757492,-95.365791,0.000,Houston,77002-6308,86.0,66.0,29.84,8.0,ESE,9.2,0.0,Clear,False,False,False,False,False,True,False,True,Day
1,Source2,2,2016-06-14 20:26:55,2016-06-14 21:12:35,29.821486,-95.368080,0.000,Houston,77022-4842,84.2,70.0,29.84,8.0,ESE,9.2,0.0,Clear,False,False,False,False,False,False,True,False,Night
2,Source1,2,2016-06-17 15:44:51,2016-06-17 21:44:51,29.784610,-95.617360,0.940,Houston,77079,93.0,50.0,29.94,10.0,SSE,10.4,0.0,Partly Cloudy,False,False,True,True,False,False,False,False,Day
3,Source1,3,2016-06-17 15:53:54,2016-06-17 21:53:54,29.784750,-95.683090,1.959,Houston,77094,93.0,50.0,29.94,10.0,SSE,10.4,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Day
4,Source2,2,2016-06-21 09:27:32,2016-06-21 10:42:32,29.859404,-95.356445,0.000,Houston,77093-4446,84.9,69.0,30.14,10.0,WNW,4.6,0.0,Scattered Clouds,False,True,False,False,False,False,True,False,Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
503343,Source1,2,2023-03-20 07:06:30,2023-03-20 09:29:00,25.811510,-80.199829,3.469,Miami,33127,60.0,96.0,30.08,10.0,NW,9.0,0.0,Cloudy,False,False,False,True,False,False,False,False,Night
503344,Source1,2,2023-03-26 16:11:30,2023-03-26 17:01:13,25.787840,-80.209907,0.386,Miami,33136,86.0,57.0,30.04,10.0,SSE,8.0,0.0,Mostly Cloudy,False,False,False,False,False,False,False,False,Day
503345,Source1,2,2023-03-26 16:29:39,2023-03-26 21:29:38,25.787316,-80.210999,0.800,Miami,33136,84.0,61.0,30.03,10.0,SE,12.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Day
503346,Source1,2,2023-03-30 06:41:00,2023-03-30 09:24:30,25.897225,-80.380845,0.701,Miami,33178,73.0,90.0,30.07,10.0,N,3.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Night


## Feature Engineering

In [ ]:
# Adding a duration of accidents in minutes

d2['Duration_Minutes'] = (d2['End_Time'] - d2['Start_Time']).dt.total_seconds() / 60

# Changing Start_Time and End_Time to separate date and time columns 
d2['Start_Date'] = d2['Start_Time'].dt.date
d2['Start_Clock_Time'] = d2['Start_Time'].dt.time

d2["End_Date"] = d2["End_Time"].dt.date
d2["End_Clock_Time"] = d2["End_Time"].dt.time

In [ ]:
# Extracting more relevant predictors from the start time, such as hour, day of week, weekend, etc
d2['Hour'] = d2['Start_Time'].dt.hour
d2["Day_of_Week"] = d2['Start_Time'].dt.dayofweek
d2['Month'] = d2['Start_Time'].dt.month
d2['Weekend'] = d2['Start_Time'].dt.weekday >= 5


In [109]:
import holidays

# create a US holidays object
us_holidays = holidays.UnitedStates()
# create a Holiday column that indicates whether the date is a holiday
d2['Holiday'] = d2['Start_Date'].apply(lambda x: x in us_holidays).astype(int)
d2['Holiday_Name'] = d2['Start_Time'].dt.date.apply(lambda x: us_holidays.get(x))
d2["Holiday_Name"].fillna("Not Holiday", inplace=True)

/var/folders/x7/sl4zn7qx64s6wq122_6jw9_00000gn/T/ipykernel_5167/1919409221.py:8: FutureWarning:

A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.





In [110]:
d2[d2['Holiday'] == 1]

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,Zipcode,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset,Duration_Minutes,Start_Date,Start_Clock_Time,End_Date,End_Clock_Time,Holiday,Holiday_Name
3432848,Source1,3,2016-07-04 00:17:22,2016-07-04 06:17:22,29.777540,-95.371850,0.932,Houston,77007,82.4,79.0,29.95,10.0,ESE,4.6,0.0,Clear,False,False,False,False,False,False,False,False,Night,360.000000,2016-07-04,00:17:22,2016-07-04,06:17:22,1,Independence Day
3432851,Source1,4,2016-07-04 02:02:40,2016-07-04 08:02:40,29.693499,-95.525209,2.248,Houston,77074,82.4,79.0,29.95,10.0,Calm,4.6,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Night,360.000000,2016-07-04,02:02:40,2016-07-04,08:02:40,1,Independence Day
3432855,Source1,3,2016-07-04 04:46:08,2016-07-04 10:46:08,29.691730,-95.283620,0.847,Houston,77017,81.0,88.0,29.96,10.0,South,5.8,0.0,Partly Cloudy,False,False,False,True,False,False,False,False,Night,360.000000,2016-07-04,04:46:08,2016-07-04,10:46:08,1,Independence Day
302429,Source2,2,2016-07-04 06:00:56,2016-07-04 06:47:55,29.715075,-95.571510,0.000,Houston,77072,80.1,87.0,29.95,10.0,South,10.4,0.0,Clear,False,False,False,False,False,False,False,False,Night,46.983333,2016-07-04,06:00:56,2016-07-04,06:47:55,1,Independence Day
302432,Source2,2,2016-07-04 06:36:44,2016-07-04 07:06:44,29.737444,-95.510094,0.000,Houston,77063-2901,80.6,89.0,29.94,7.0,ESE,5.8,0.0,Clear,False,True,False,False,False,False,False,True,Day,30.000000,2016-07-04,06:36:44,2016-07-04,07:06:44,1,Independence Day
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4452492,Source1,2,2023-01-16 21:01:30,2023-01-16 21:31:00,25.833118,-80.241320,0.131,Miami,33147-7799,60.0,60.0,30.09,10.0,CALM,0.0,0.0,Fair,False,False,False,False,False,False,False,False,Night,29.500000,2023-01-16,21:01:30,2023-01-16,21:31:00,1,Martin Luther King Jr. Day
4669341,Source1,2,2023-01-16 21:14:44,2023-01-16 22:32:02,25.807459,-80.292632,0.018,Miami,33166,63.0,65.0,30.09,10.0,CALM,0.0,0.0,Fair,False,False,False,False,False,True,False,False,Night,77.300000,2023-01-16,21:14:44,2023-01-16,22:32:02,1,Martin Luther King Jr. Day
5356519,Source1,2,2023-01-16 21:29:57,2023-01-17 01:56:06,25.759270,-80.265129,0.128,Miami,33134-2725,61.0,70.0,30.09,10.0,CALM,0.0,0.0,Fair,False,False,False,False,False,False,False,False,Night,266.150000,2023-01-16,21:29:57,2023-01-17,01:56:06,1,Martin Luther King Jr. Day
5149428,Source1,2,2023-01-16 22:17:13,2023-01-16 23:35:03,25.699823,-80.462600,0.092,Miami,33193,55.0,96.0,30.09,10.0,CALM,0.0,0.0,Fair,False,True,False,False,False,False,False,False,Night,77.833333,2023-01-16,22:17:13,2023-01-16,23:35:03,1,Martin Luther King Jr. Day


### Converting Data to Numerical

In [111]:
#pd.set_option('display.max_columns', None)

In [112]:
# One Hot Encoding the cities column
cities = pd.get_dummies(d2['City'], prefix='City')
d2 = pd.concat([d2, cities], axis=1)

In [ ]:
# Since there is too many weather conditions, must consolidate them before doing one hot encoding

weather_conditions_map = {
    'Fair': 'Clear',
    'Clear': 'Clear',
    'Fair / Windy': 'Clear',

    'Mostly Cloudy': 'Cloudy',
    'Partly Cloudy': 'Cloudy',
    'Cloudy': 'Cloudy',
    'Overcast': 'Cloudy',
    'Scattered Clouds': 'Cloudy',
    'Mostly Cloudy / Windy': 'Cloudy',
    'Cloudy / Windy': 'Cloudy',
    'Partly Cloudy / Windy': 'Cloudy',

    'Light Rain': 'Rain/Drizzle',
    'Rain': 'Rain/Drizzle',
    'Heavy Rain': 'Rain/Drizzle',
    'Light Rain / Windy': 'Rain/Drizzle',
    'Light Drizzle': 'Rain/Drizzle',
    'Drizzle': 'Rain/Drizzle',
    'Rain / Windy': 'Rain/Drizzle',
    'Heavy Rain / Windy': 'Rain/Drizzle',
    'Drizzle and Fog': 'Rain/Drizzle',
    'Showers in the Vicinity': 'Rain/Drizzle',
    'Light Rain Showers': 'Rain/Drizzle',
    'Rain Showers': 'Rain/Drizzle',

    'Thunder in the Vicinity': 'Stormy',
    'Thunder': 'Stormy',
    'T-Storm': 'Stormy',
    'Heavy T-Storm': 'Stormy',
    'Thunderstorm': 'Stormy',
    'Light Rain with Thunder': 'Stormy',
    'Light Thunderstorms and Rain': 'Stormy',
    'Heavy Thunderstorms and Rain': 'Stormy',
    'Thunderstorms and Rain': 'Stormy',
    'Heavy T-Storm / Windy': 'Stormy',
    'T-Storm / Windy': 'Stormy',
    'Thunder / Windy': 'Stormy',
    'Squalls / Windy': 'Stormy',
    'Funnel Cloud': 'Stormy',

    'Haze': 'Visibility Issues',
    'Fog': 'Visibility Issues',
    'Shallow Fog': 'Visibility Issues',
    'Mist': 'Visibility Issues',
    'Patches of Fog': 'Visibility Issues',
    'Smoke': 'Visibility Issues',
    'Haze / Windy': 'Visibility Issues',

    'Light Snow': 'Snow/Ice',
    'Snow': 'Snow/Ice',
    'Light Snow and Sleet / Windy': 'Snow/Ice',
    'Wintry Mix': 'Snow/Ice',
    'Wintry Mix / Windy': 'Snow/Ice',
    'Light Sleet': 'Snow/Ice',
    'Heavy Sleet': 'Snow/Ice',
    'Snow and Sleet / Windy': 'Snow/Ice',
    'Heavy Snow': 'Snow/Ice',
    'Light Ice Pellets': 'Snow/Ice',
    'Light Freezing Rain': 'Snow/Ice',
    'Freezing Rain / Windy': 'Snow/Ice',

    'Widespread Dust': 'Dust/Windy',
    'Blowing Dust': 'Dust/Windy',
    'Blowing Dust / Windy': 'Dust/Windy',
    'Duststorm': 'Dust/Windy',
    
}

d2["Weather_Condition_Grouped"] = d2["Weather_Condition"].map(weather_conditions_map).fillna("Other")

In [114]:
conditions = pd.get_dummies(d2["Weather_Condition_Grouped"], prefix="Weather")
d2 = pd.concat([d2, conditions], axis=1)

In [ ]:
# Converting Sunrise_Sunset to a binary variable for day and night
day_map = {
    'Night': 0,
    'Day': 1
}

d2['Day/Night'] = d2['Sunrise_Sunset'].map(day_map)

### Zone ID Creation

In [117]:
import h3

# Uses Uber's geospatial indexing library to convert lat/lng to hexagonal zones for better grouping
def get_hex_id(row):
    lat = row['Start_Lat']
    lng = row['Start_Lng']
    return h3.latlng_to_cell(lat, lng, res=7)
    

# Create the Zone_ID to plot the longitude and latitude into hexagonal zones
d2['Zone_ID'] = d2.apply(get_hex_id, axis=1)

# Making the zone into a simple integer
d2['Zone_Int_ID'], _ = pd.factorize(d2['Zone_ID'])

In [118]:
import plotly.express as px
import h3

# Aggregate accidents by the different zones in Houston - change to different cities for diff results
houston_hex = d2[d2['City_Houston'] == 1].groupby('Zone_ID').size().reset_index(name='Accident_Count')

# Using GeoJSON to plot the hexagonal zones on the map with Plotly
features = []
for zone in houston_hex['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': features}

# Plotting the hexagonal zones with accident counts 
fig = px.choropleth_map(
    houston_hex,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Accident_Count',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    center={'lat': 29.7604, 'lon': -95.3698}, # Coordinates for Houston
    # center={'lat': 25.7617, 'lon': -80.1918} # Coordinates for Miami
    # center={'lat': 34.0522, 'lon': -118.2437} # Coordinates for Los Angeles
    labels={'Accident_Count': 'Accidents'}
)

fig.update_layout()
fig.show()

In [121]:
d2

,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,Distance(mi),City,Zipcode,Temperature(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Crossing,Give_Way,Junction,Railway,Station,Stop,Traffic_Signal,Sunrise_Sunset,Duration_Minutes,Start_Date,Start_Clock_Time,End_Date,End_Clock_Time,Holiday,Holiday_Name,City_Houston,City_Los Angeles,City_Miami,Weather_Condition_Grouped,Weather_Clear,Weather_Cloudy,Weather_Dust/Windy,Weather_Rain/Drizzle,Weather_Snow/Ice,Weather_Stormy,Weather_Visibility Issues,Day/Night,Zone_ID,Zone_Int_ID
300351,Source2,2,2016-06-14 20:21:49,2016-06-14 21:07:49,29.757492,-95.365791,0.000,Houston,77002-6308,86.0,66.0,29.84,8.0,ESE,9.2,0.0,Clear,False,False,False,False,False,True,False,True,Day,46.000000,2016-06-14,20:21:49,2016-06-14,21:07:49,0,Not Holiday,True,False,False,Clear,True,False,False,False,False,False,False,1,87446ca98ffffff,0
300350,Source2,2,2016-06-14 20:26:55,2016-06-14 21:12:35,29.821486,-95.368080,0.000,Houston,77022-4842,84.2,70.0,29.84,8.0,ESE,9.2,0.0,Clear,False,False,False,False,False,False,True,False,Night,45.666667,2016-06-14,20:26:55,2016-06-14,21:12:35,0,Not Holiday,True,False,False,Clear,True,False,False,False,False,False,False,0,87446c304ffffff,1
3431765,Source1,2,2016-06-17 15:44:51,2016-06-17 21:44:51,29.784610,-95.617360,0.940,Houston,77079,93.0,50.0,29.94,10.0,SSE,10.4,0.0,Partly Cloudy,False,False,True,True,False,False,False,False,Day,360.000000,2016-06-17,15:44:51,2016-06-17,21:44:51,0,Not Holiday,True,False,False,Cloudy,False,True,False,False,False,False,False,1,87446caecffffff,2
3431759,Source1,3,2016-06-17 15:53:54,2016-06-17 21:53:54,29.784750,-95.683090,1.959,Houston,77094,93.0,50.0,29.94,10.0,SSE,10.4,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Day,360.000000,2016-06-17,15:53:54,2016-06-17,21:53:54,0,Not Holiday,True,False,False,Cloudy,False,True,False,False,False,False,False,1,87446ca58ffffff,3
300360,Source2,2,2016-06-21 09:27:32,2016-06-21 10:42:32,29.859404,-95.356445,0.000,Houston,77093-4446,84.9,69.0,30.14,10.0,WNW,4.6,0.0,Scattered Clouds,False,True,False,False,False,False,True,False,Day,75.000000,2016-06-21,09:27:32,2016-06-21,10:42:32,0,Not Holiday,True,False,False,Cloudy,False,True,False,False,False,False,False,1,87446c300ffffff,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3652181,Source1,2,2023-03-20 07:06:30,2023-03-20 09:29:00,25.811510,-80.199829,3.469,Miami,33127,60.0,96.0,30.08,10.0,NW,9.0,0.0,Cloudy,False,False,False,True,False,False,False,False,Night,142.500000,2023-03-20,07:06:30,2023-03-20,09:29:00,0,Not Holiday,False,False,True,Cloudy,False,True,False,False,False,False,False,0,8744a1172ffffff,582
3649977,Source1,2,2023-03-26 16:11:30,2023-03-26 17:01:13,25.787840,-80.209907,0.386,Miami,33136,86.0,57.0,30.04,10.0,SSE,8.0,0.0,Mostly Cloudy,False,False,False,False,False,False,False,False,Day,49.716667,2023-03-26,16:11:30,2023-03-26,17:01:13,0,Not Holiday,False,False,True,Cloudy,False,True,False,False,False,False,False,1,8744a1176ffffff,558
3644380,Source1,2,2023-03-26 16:29:39,2023-03-26 21:29:38,25.787316,-80.210999,0.800,Miami,33136,84.0,61.0,30.03,10.0,SE,12.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Day,299.983333,2023-03-26,16:29:39,2023-03-26,21:29:38,0,Not Holiday,False,False,True,Cloudy,False,True,False,False,False,False,False,1,8744a1176ffffff,558
3663772,Source1,2,2023-03-30 06:41:00,2023-03-30 09:24:30,25.897225,-80.380845,0.701,Miami,33178,73.0,90.0,30.07,10.0,N,3.0,0.0,Partly Cloudy,False,False,False,False,False,False,False,False,Night,163.500000,2023-03-30,06:41:00,2023-03-30,09:24:30,0,Not Holiday,False,False,True,Cloudy,False,True,False,False,False,False,False,0,8744a1b99ffffff,618


In [ ]:
# Dropping any unnecessary columns for modelling - start times and zipcodes and lat/lng are too sparse
modelling_df = d2.drop(columns=['Source','Start_Time', 'End_Time', 'City', 'Zipcode', 'Start_Lat', 'Start_Lng', 'Wind_Direction', 'Weather_Condition', 'Weather_Condition_Grouped', 'Holiday_Name', 'Zone_ID', "Start_Date", "Start_Clock_Time", "End_Date", "End_Clock_Time", "Sunrise_Sunset"])

In [ ]:
modelling_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 503348 entries, 300351 to 3670856
Data columns (total 34 columns):
 #   Column                     Non-Null Count   Dtype  
---  ------                     --------------   -----  
 0   Severity                   503348 non-null  int64  
 1   Distance(mi)               503348 non-null  float64
 2   Temperature(F)             503348 non-null  float64
 3   Humidity(%)                503348 non-null  float64
 4   Pressure(in)               503348 non-null  float64
 5   Visibility(mi)             503348 non-null  float64
 6   Wind_Speed(mph)            503348 non-null  float64
 7   Precipitation(in)          503348 non-null  float64
 8   Amenity                    503348 non-null  bool   
 9   Crossing                   503348 non-null  bool   
 10  Give_Way                   503348 non-null  bool   
 11  Junction                   503348 non-null  bool   
 12  Railway                    503348 non-null  bool   
 13  Station                    5